LLaVA-7B Fine-tuning for DocVQA (SP-DocVQA)
===========================================

This notebook walks you through:
- Installing dependencies (Windows-friendly)
- Converting SP-DocVQA JSON to LLaVA chat JSONL
- Fine-tuning `llava-hf/llava-1.5-7b-hf` with LoRA 
- Quick inference check with the trained adapter

In [ ]:
# Fixed config for Kaggle
import os, torch

train_parquet = "/kaggle/input/docvqa-minidata-1200-samples/train-00000-of-00057.parquet"
val_parquet = "/kaggle/input/docvqa-minidata-1200-samples/validation-00000-of-00008.parquet"

model_id = "llava-hf/llava-1.5-7b-hf"  # set to a Kaggle dataset path if internet is disabled
output_dir = "/kaggle/working/outputs/llava-7b-docvqa"
os.makedirs(output_dir, exist_ok=True)

# Training hyperparameters
supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_bf16 = True if supports_bf16 else False
use_fp16 = not use_bf16
use_4bit = True  # QLoRA on Kaggle (T4/A100)
per_device_train_batch_size = 1
gradient_accumulation_steps = 16
epochs = 1
learning_rate = 2e-4
gradient_checkpointing = True

print("Train Parquet:", train_parquet)
print("Val Parquet:", val_parquet)
print("Output dir:", output_dir)
print("bf16:", use_bf16, "fp16:", use_fp16, "4bit:", use_4bit)



In [ ]:
# Utils: image loader and feature builder for Parquet
from io import BytesIO
from typing import Any, Dict, List
from PIL import Image

def image_from_any(value: Any) -> Image.Image:
	if isinstance(value, Image.Image):
		img = value
	elif isinstance(value, dict) and "bytes" in value:
		data = value["bytes"]
		if not isinstance(data, (bytes, bytearray)):
			data = bytes(data)
		img = Image.open(BytesIO(data))
	elif isinstance(value, list) and len(value) > 0:
		first = value[0]
		return image_from_any(first)
	elif isinstance(value, (bytes, bytearray)):
		img = Image.open(BytesIO(value))
	else:
		raise ValueError(f"Unsupported image type: {type(value)}")
	if img.mode != "RGB":
		img = img.convert("RGB")
	return img

def build_messages_from_qa(question: str, answer: str) -> List[Dict[str, Any]]:
	return [
		{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": (question or "").strip()}]},
		{"role": "assistant", "content": [{"type": "text", "text": (answer or "").strip()}]},
	]



In [ ]:
# Prepare datasets and feature mapping
from datasets import load_dataset
from transformers import AutoProcessor, AutoTokenizer

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
	tokenizer.pad_token = tokenizer.eos_token

question_field = "question"
answers_field = "answers"
image_field = "image"


def prepare_features(rec):
	image = image_from_any(rec[image_field])
	ans_val = rec.get(answers_field, "")
	answer = (ans_val[0] if isinstance(ans_val, (list, tuple)) and ans_val else ans_val) or ""
	messages = build_messages_from_qa(rec.get(question_field, ""), answer)

	prompt_text = tokenizer.apply_chat_template(messages[:1], add_generation_prompt=True, tokenize=False)
	full_text = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)

	model_inputs = processor(text=[full_text], images=[image], return_tensors="pt")
	input_ids = model_inputs["input_ids"][0]
	attention_mask = model_inputs["attention_mask"][0]
	pixel_values = model_inputs["pixel_values"][0]

	prompt_ids = tokenizer(prompt_text, add_special_tokens=False, return_tensors="pt").input_ids[0]
	labels = input_ids.clone()
	num_prompt_tokens = min(prompt_ids.shape[0], labels.shape[0])
	labels[:num_prompt_tokens] = -100
	labels = labels.masked_fill(attention_mask == 0, -100)

	return {
		"input_ids": input_ids.tolist(),
		"attention_mask": attention_mask.tolist(),
		"labels": labels.tolist(),
		"pixel_values": pixel_values,
	}

raw = load_dataset("parquet", data_files={"train": train_parquet, "validation": val_parquet})
keep_cols = {image_field, question_field, answers_field}

ds_proc = {}
for split_name, d in raw.items():
	ds_proc[split_name] = d.map(
		prepare_features,
		remove_columns=[c for c in d.column_names if c not in keep_cols],
		desc=f"Tokenizing {split_name}",
	)

train_dataset = ds_proc["train"]
eval_dataset = ds_proc.get("validation")
print(train_dataset)
print(eval_dataset)



In [ ]:
# Data collator
import torch
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class LlavaCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:

        # FIX: Convert pixel_values → tensor (đúng vòng for)
        pixel_values = [
            f["pixel_values"] if isinstance(f["pixel_values"], torch.Tensor)
            else torch.tensor(f["pixel_values"], dtype=torch.float32)
            for f in features
        ]
        pixel_values = torch.stack(pixel_values, dim=0)

        # Token fields
        input_ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        attention_mask = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features]
        labels = [torch.tensor(f["labels"], dtype=torch.long) for f in features]

        # Pad input_ids + attention_mask
        batch_input = self.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True,
            return_tensors="pt"
        )

        # Pad labels
        batch_labels = self.tokenizer.pad(
            {"input_ids": labels},
            padding=True,
            return_tensors="pt"
        )["input_ids"]

        # Mask pad → -100
        batch_labels = batch_labels.masked_fill(batch_input["attention_mask"] == 0, -100)

        return {
            "input_ids": batch_input["input_ids"],
            "attention_mask": batch_input["attention_mask"],
            "labels": batch_labels,
            "pixel_values": pixel_values,
        }


collator = LlavaCollator(tokenizer=tokenizer)


In [ ]:
from transformers import BitsAndBytesConfig
from transformers import LlavaForConditionalGeneration, AutoProcessor
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# Model (QLoRA optional)
quant_config = None
if use_4bit:
	try:
		quant_config = BitsAndBytesConfig(
			load_in_4bit=True,
			bnb_4bit_quant_type="nf4",
			bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
			bnb_4bit_use_double_quant=True,
		)
	except Exception as e:
		print("bitsandbytes unavailable, falling back to full precision LoRA:", e)
		quant_config = None

# Avoid meta tensor move issues: load without device_map auto and without low_cpu_mem_usage
model = LlavaForConditionalGeneration.from_pretrained(
	model_id,
	quantization_config=quant_config,
	torch_dtype=torch.bfloat16 if use_bf16 else (torch.float16 if use_fp16 else None),
	low_cpu_mem_usage=False,
	device_map=None,
	trust_remote_code=True,
)
# Move fully to CUDA
model.to("cuda")

model.config.use_cache = False
if gradient_checkpointing:
	model.gradient_checkpointing_enable()
if quant_config is not None:
	model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
	r=64,
	lora_alpha=16,
	lora_dropout=0.05,
	bias="none",
	target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
	task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_ratio=0.0,
    weight_decay=0.0,
    logging_steps=1,
    logging_first_step=True,
    # Nếu version cũ không hỗ trợ evaluation_strategy/save_strategy
    # có thể bỏ 2 dòng dưới
    # evaluation_strategy="no",
    # save_strategy="no",
    max_steps=1,           # chỉ chạy 1 bước để test
    learning_rate=learning_rate,
    optim="adamw_torch",
    fp16=use_fp16,
    bf16=use_bf16,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()

# Save outputs
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"Saved to: {output_dir}")


In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
item = train_dataset[0]
type(item["pixel_values"]), item["pixel_values"].__class__
